# Redução de Dimensionalidade das Métricas Locais via PCA

In [67]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind


from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from scipy.stats import mannwhitneyu
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

from scipy.stats import ttest_ind

import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_dark"


### Carregando o dataset

In [68]:
DATASET_PATH = "../data/processed/dataset_features.parquet"

df_dataset = pd.read_parquet(DATASET_PATH)
df_dataset.shape
df_filtered = df_dataset.columns[df_dataset.columns.str.contains("age", case=False)]
df_filtered



Index(['Age'], dtype='object')

In [69]:
df_dataset.head()

,degree_0,degree_1,degree_2,degree_3,degree_4,degree_5,degree_6,degree_7,degree_8,degree_9,...,SKID_Diagnoses 1,SKID_Diagnoses 2,Comments_SKID_assessment,Hamilton_Scale,BSL23_sumscore,BSL23_behavior,AUDIT,Standard_Alcoholunits_Last_28days,Alcohol_Dependence_In_1st-3rd_Degree_relative,Relationship_Status
0,39449.859168,39102.152342,40387.630997,37566.960373,39408.112994,38719.391893,40815.072973,41275.499305,39649.516506,40196.211832,...,"#CODE:296.26 #DESC:Major Depressive Disorder,...",None,past subclinical dperession,0.0,7.0,1.0,7.0,29.5,No,No
1,41513.435372,41327.513954,40750.435276,40491.263384,41233.158432,41146.017954,40730.430082,41351.132165,39371.224746,39566.056291,...,#CODE:V71.09 #DESC:No Diagnosis or Condition ...,None,None,4.0,6.0,2.0,11.0,53.5,Yes (9),No
2,40727.834359,40001.600624,40267.149863,39308.151269,39753.137398,39336.794272,40117.319684,40446.206489,38441.008195,38531.272048,...,#CODE:V71.09 #DESC:No Diagnosis or Condition ...,None,None,4.0,6.0,2.0,11.0,53.5,Yes (9),No
3,38401.305380,38333.117784,37902.877026,37959.206795,38497.084816,37761.467224,37754.236560,37956.444283,36880.581031,37203.264413,...,#CODE:V71.09 #DESC:No Diagnosis or Condition ...,None,None,4.0,3.0,1.0,3.0,5,No,Yes
4,42558.276311,42950.141781,42023.976342,40857.941940,40478.291440,41090.310476,42519.571111,42339.443250,40755.136885,42859.521462,...,#CODE:V71.09 #DESC:No Diagnosis or Condition ...,None,None,4.0,3.0,1.0,3.0,5,No,Yes


### Seleção Métricas Locais

In [70]:
NODE_FEATURE_PREFIXES = [
    "degree_",
    "clustering_",
    "betweenness_",
    "local_efficiency_",
    "hub_frequency_",
    "weighted_degree_"
]

NODE_FEATURES = [
    c for c in df_dataset.columns
    if any(c.startswith(p) for p in NODE_FEATURE_PREFIXES)
]

NODE_GROUPS = {
    "degree": [c for c in df_dataset.columns if c.startswith("degree_")],
    "clustering": [c for c in df_dataset.columns if c.startswith("clustering_")],
    "betweenness": [c for c in df_dataset.columns if c.startswith("betweenness_")],
    "local_efficiency": [c for c in df_dataset.columns if c.startswith("local_efficiency_")],
    "weighted_degree": [c for c in df_dataset.columns if c.startswith("weighted_degree_")]
}


len(NODE_FEATURES)


366

In [71]:
X = df_dataset[NODE_FEATURES].values
y = df_dataset["condition"].values

### Padronização das Feaures

Substituindo Nan por zero:

In [72]:
X = np.nan_to_num(X, nan=0.0)

In [73]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [74]:
X_scaled.mean(axis=0)[:5], X_scaled.std(axis=0)[:5]

(array([-1.99705572e-15, -1.66466167e-15, -1.47491447e-15,  1.61486985e-16,
         9.66230463e-16]),
 array([1., 1., 1., 1., 1.]))

### PCA exploratório

In [75]:
np.isnan(X).sum()


np.int64(0)

In [76]:
nan_per_feature = pd.Series(np.isnan(X).sum(axis=0), index=NODE_FEATURES)
nan_per_feature[nan_per_feature > 0].sort_values(ascending=False)


Series([], dtype: int64)

In [77]:
pca_full = PCA()
X_pca_full = pca_full.fit_transform(X_scaled)

explained_var = pca_full.explained_variance_ratio_
cum_var = np.cumsum(explained_var)


In [78]:
df_var = pd.DataFrame({
    "component": np.arange(1, len(explained_var) + 1),
    "explained_variance": explained_var,
    "cumulative_variance": cum_var
})


In [79]:
df_var

,component,explained_variance,cumulative_variance
0,1,5.245205e-01,0.524521
1,2,8.573967e-02,0.610260
2,3,3.187226e-02,0.642132
3,4,2.968941e-02,0.671822
4,5,2.564496e-02,0.697467
...,...,...,...
160,161,6.133566e-09,1.000000
161,162,3.894672e-09,1.000000
162,163,2.612905e-09,1.000000
163,164,1.738400e-09,1.000000


In [80]:
fig = px.line(
    df_var,
    x="component",
    y="cumulative_variance",
    markers=True,
    title="Variância Acumulada Explicada pelo PCA"
)
fig.add_hline(y=0.8, line_dash="dash", annotation_text="80%")
fig.add_hline(y=0.9, line_dash="dash", annotation_text="90%")
fig.show()


Observa-se que aproximadamente 14 componentes principais são suficientes para explicar cerca de 80% da variância total dos dados, enquanto 29 componentes explicam cerca de 90%, indicando elevada redundância entre as métricas de rede.

### PCA com 2 componentes

In [81]:
NODE_FEATURE_PREFIXES = [
    "degree_",
    "clustering_",
    "betweenness_",
    "local_efficiency_",
    "hub_frequency_",
    "weighted_degree_"
]

# Create a dictionary where keys are the metric names (e.g., 'degree')
NODE_FEATURES_BY_TYPE = {
    prefix.rstrip("_"): [c for c in NODE_FEATURES if c.startswith(prefix)]
    for prefix in NODE_FEATURE_PREFIXES
}



In [82]:
pca_2 = PCA(n_components=2)
X_pca_2 = pca_2.fit_transform(X_scaled)

df_pca = pd.DataFrame(
    X_pca_2,
    columns=["PC1", "PC2"]
)
df_pca["condition"] = y


In [83]:
fig = px.scatter(
    df_pca,
    x="PC1",
    y="PC2",
    color="condition",
    title="Projeção PCA das Métricas Locais (2 Componentes)",
    opacity=0.8
)
fig.show()


Aqui você responde visualmente:

existe separação EO × EC? Não há separação clara entre EO e EC

existe sobreposição? Sim

clusters? Existem regiões de maior densidade, mas não clusters bem definidos.

A projeção das métricas locais no espaço definido pelos dois primeiros
componentes principais não evidenciou uma separação clara entre as
condições EO e EC. Observa-se uma ampla sobreposição entre os pontos
correspondentes às duas condições experimentais, indicando que os
padrões dominantes de variabilidade capturados pelos componentes
principais são compartilhados entre os grupos.

Embora seja possível identificar regiões de maior densidade de pontos,
não foram observados clusters bem definidos ou fronteiras visuais que
permitam distinguir claramente as condições experimentais no espaço
PC1–PC2.


### Contribuição das features

In [84]:
loadings = pd.DataFrame(
    pca_2.components_.T,
    index=NODE_FEATURES,
    columns=["PC1", "PC2"]
)


In [85]:
loadings

,PC1,PC2
degree_0,0.077675,-0.034629
degree_1,0.076115,-0.036981
degree_2,0.078581,-0.044435
degree_3,0.073001,-0.053006
degree_4,0.070616,-0.054444
...,...,...
weighted_degree_56,0.050196,-0.067618
weighted_degree_57,0.050254,-0.091723
weighted_degree_58,0.042000,-0.108827
weighted_degree_59,0.030978,-0.142661


In [86]:
loadings.abs().sort_values("PC1", ascending=False).head(10)

,PC1,PC2
weighted_degree_46,0.082558,0.017579
degree_46,0.082558,0.017579
weighted_degree_16,0.082146,0.031428
degree_16,0.082146,0.031428
weighted_degree_6,0.082016,0.028846
degree_6,0.082016,0.028846
weighted_degree_10,0.081638,0.019646
degree_10,0.081638,0.019646
weighted_degree_52,0.081264,0.048791
degree_52,0.081264,0.048791


In [87]:
loadings.abs().sort_values("PC2", ascending=False).head(10)

,PC1,PC2
clustering_20,0.056906,0.150630
local_efficiency_59,0.025072,0.142863
degree_59,0.030978,0.142661
weighted_degree_59,0.030978,0.142661
clustering_59,0.031085,0.135050
local_efficiency_60,0.021359,0.124430
degree_60,0.025563,0.122206
weighted_degree_60,0.025563,0.122206
clustering_40,0.070889,0.120346
clustering_15,0.071363,0.120328


Quais métricas e nós mais contribuem para cada componente?

PC1 separa redes mais ou menos conectadas globalmente, enquanto PC2 captura diferenças na organização local e eficiência estrutural.

### PCA com mais componentes - Para ML

In [88]:
pca_n = PCA(n_components=0.9)  # mantém 90% da variância
X_pca_n = pca_n.fit_transform(X_scaled)

X_pca_n.shape

(165, 29)

Para as análises de aprendizado de máquina, foi utilizado PCA com retenção de 90% da variância, resultando em 29 componentes principais.

A aplicação do PCA sobre as métricas locais permitiu reduzir significativamente
a dimensionalidade do conjunto de dados, preservando a maior parte da variância
original.

A projeção bidimensional indica padrões de organização distintos entre as
condições experimentais, embora com sobreposição parcial, sugerindo que métodos
não lineares ou classificadores supervisionados podem capturar diferenças mais sutis.

Os loadings indicam que métricas associadas a determinados nós apresentam maior
contribuição para os componentes principais, reforçando a relevância da análise
local da conectividade funcional.


-----

### Teste Estatístico

In [89]:
pc1_eo = df_pca.loc[df_pca.condition == "EO", "PC1"]
pc1_ec = df_pca.loc[df_pca.condition == "EC", "PC1"]

ttest_ind(pc1_eo, pc1_ec)

TtestResult(statistic=np.float64(0.4263767728516219), pvalue=np.float64(0.6703993448759286), df=np.float64(162.0))

In [90]:

mannwhitneyu(pc1_eo, pc1_ec, alternative="two-sided")


MannwhitneyuResult(statistic=np.float64(3488.0), pvalue=np.float64(0.6797957470870699))

In [91]:
px.box(df_pca, x="condition", y="PC1")


In [92]:
px.box(df_pca, x="condition", y="PC2")


In [93]:
df_pca = df_pca[df_pca["condition"].isin(["EO", "EC"])]

In [94]:


X = df_pca[["PC1", "PC2"]]
y = df_pca["condition"]

model = LogisticRegression()
scores = cross_val_score(model, X, y, cv=5)

scores.mean()


np.float64(0.47556818181818183)

As duas primeiras componentes principais não apresentaram capacidade discriminativa entre as condições EO e EC, uma vez que a acurácia média obtida foi próxima ao nível de acaso (≈50%).

In [95]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

X = df_dataset[NODE_FEATURES]
y = df_dataset["condition"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = LogisticRegression(
    penalty="l1",
    solver="saga",
    max_iter=1000
)


c:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\.venv\Lib\site-packages\sklearn\utils\extmath.py:1207: RuntimeWarning:

invalid value encountered in divide

c:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\.venv\Lib\site-packages\sklearn\utils\extmath.py:1212: RuntimeWarning:

invalid value encountered in divide

c:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\.venv\Lib\site-packages\sklearn\utils\extmath.py:1236: RuntimeWarning:

invalid value encountered in divide



In [96]:
scores = cross_val_score(model, X_scaled, y, cv=5)
scores.mean()


c:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\.venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning:

The least populated class in y has only 1 members, which is less than n_splits=5.

c:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning:

'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.

c:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning:

Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.

c:\Users\Anne\OneDrive\Documentos\Goo

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\.venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\.venv\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py